In [1]:
# libraries
import os
from dotenv import load_dotenv
from openai import OpenAI

import gradio as gr

In [2]:
# set up environment

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("API Keys not found")
elif not api_key.startswith('sk-proj-'):
    print("API key found but wrong format")
elif api_key.strip() != api_key:
    print("API key found but contain unnecessary space")
else:
    print("API Key found and in use")


openai = OpenAI()
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

API Key found and in use


In [3]:

system_message = "You are a helpful assistant."

def message_gpt(prompt):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': prompt}
    ]

    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages)

    return response.choices[0].message.content


message_gpt('How is the current president of USA?')

'As of October 2023, the President of the United States is Joe Biden. He took office on January 20, 2021. If you need more specific information or updates regarding his presidency or policies, feel free to ask!'

## User Interface - Data Science UI

In [4]:
def shout(text):
    print(f"Shout has been called with input {text}")

    return text.upper()

shout('hello')

Shout has been called with input hello


'HELLO'

In [ ]:
gr.Interface(fn=shout, inputs='textbox', outputs='textbox', flagging_mode='never').launch(inbrowser=True, auth=('ibusah', 'great12@'))


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input howdy


In [ ]:
# adding more fields
message_input = gr.Textbox(label='Your message:', info='Enter your message...', lines=7)
message_output = gr.Textbox(label='Response', lines=8)

view = gr.Interface(
    fn=shout,
    inputs=[message_input],
    outputs=[message_output],
    examples=['hello', 'howdy'],
    flagging_mode='never'
)

view.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input hi
Shout has been called with input howdy


In [9]:
# building the interface for the chatbot 

message_input = gr.Textbox(label='Your message:', info='Enter your message for GPT...', lines=7)
message_output = gr.Textbox(label='Response', lines=8)

view = gr.Interface(
    fn=message_gpt,
    inputs=[message_input],
    outputs=[message_output],
    examples=['How is the current president of USA?', "What's today's date?"],
    flagging_mode='never'
)

view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [10]:
system_message = "You are a helpful assistant that respond in markdown without code blocks"

message_input = gr.Textbox(label='Your message:', info='Enter your message for GPT...', lines=7)
message_output = gr.Markdown(label='Response')

view = gr.Interface(
    fn=message_gpt,
    inputs=[message_input],
    outputs=[message_output],
    examples=['Explain the Transformer architecture to layperson', "Explain the Transformer architecture to an aspiring AI engineer"],
    flagging_mode='never'
)

view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [11]:
def stream_gpt(prompt):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
          ],
        stream=True
    )    
    response = ""
    # display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [12]:
message_input = gr.Textbox(label='Your message:', info='Enter your message for GPT...', lines=7)
message_output = gr.Markdown(label='Response')

view = gr.Interface(
    fn=stream_gpt,
    title='GPT',
    inputs=[message_input],
    outputs=[message_output],
    examples=['Explain the Transformer architecture to layperson', "Explain the Transformer architecture to an aspiring AI engineer"],
    flagging_mode='never'
)

view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [13]:
def stream_ollama(prompt):
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
          ],
        stream=True
    )    
    response = ""
    # display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [14]:
message_input = gr.Textbox(label='Your message:', info='Enter your message for OLLAMA...', lines=7)
message_output = gr.Markdown(label='Response')

view = gr.Interface(
    fn=stream_ollama,
    title='OLLAMA',
    inputs=[message_input],
    outputs=[message_output],
    examples=['Explain the Transformer architecture to layperson', "Explain the Transformer architecture to an aspiring AI engineer"],
    flagging_mode='never'
)

view.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [17]:
# import time

# fruits = ["Apples", "Bananas", "Pears"]

# def come_up_with_fruit_names():
#     for fruit in fruits:
#         time.sleep(1) # thinking of a fruit
#         yield fruit

# for fruit in come_up_with_fruit_names():
#     print(fruit)

In [18]:
# adding more flavor to the UI

def stream_model(prompt, model):
    if model.upper() == 'GPT':
        result = stream_gpt(prompt)
    elif model.upper() == 'OLLAMA':
        result = stream_ollama(prompt)
    else:
        raise ValueError('Unknown Model')

    yield from result

In [19]:
message_input = gr.Textbox(label='Your message:', info='Enter your message for the LLM...', lines=7)
model_selector = gr.Dropdown(['GPT', 'Ollama'], label="Select model", value='GPT')
message_output = gr.Markdown(label='Response')

view = gr.Interface(
    fn=stream_model,
    title='LLMs',
    inputs=[message_input, model_selector],
    outputs=[message_output],
    examples=[
        ['Explain the Transformer architecture to layperson', 'GPT'], 
        ["Explain the Transformer architecture to an aspiring AI engineer", 'Ollama']
    ],
    flagging_mode='never'
)

view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
